# Install

In [1]:
!git clone https://github.com/lqb464/Sketch-to-Image-by-Pix2Pix


Cloning into 'Sketch-to-Image-by-Pix2Pix'...
remote: Enumerating objects: 2834, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 2834 (delta 117), reused 76 (delta 37), pack-reused 2644 (from 3)
Receiving objects: 100% (2834/2834), 8.51 MiB | 25.27 MiB/s, done.
Resolving deltas: 100% (1774/1774), done.


In [2]:
!pwd

/kaggle/working


In [3]:
import os
os.chdir('Sketch-to-Image-by-Pix2Pix/')


In [4]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 31.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 74.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 76.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 44.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 41.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/1

In [5]:
import os

# Đường dẫn file cần tạo
file_path = "models/unet_model.py"

# Tạo thư mục nếu chưa tồn tại
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Nội dung cần ghi
content = r'''
import torch
import torch.nn as nn
from torch.nn import init
import functools
from .base_model import BaseModel


class Identity(nn.Module):
    def forward(self, x):
        return x

def get_norm_layer(norm_type="instance"):
    if norm_type == "batch":
        norm_layer = functools.partial(nn.BatchNorm2d, affine=True, track_running_stats=True)
    elif norm_type == "syncbatch":
        norm_layer = functools.partial(nn.SyncBatchNorm, affine=True, track_running_stats=True)
    elif norm_type == "instance":
        norm_layer = functools.partial(nn.InstanceNorm2d, affine=False, track_running_stats=False)
    elif norm_type == "none":
        def norm_layer(x): return Identity()
    else:
        raise NotImplementedError("normalization layer [%s] is not found" % norm_type)
    return norm_layer

def init_weights(net, init_type="normal", init_gain=0.02):
    def init_func(m):
        classname = m.__class__.__name__
        if hasattr(m, "weight") and (classname.find("Conv") != -1 or classname.find("Linear") != -1):
            if init_type == "normal":
                init.normal_(m.weight.data, 0.0, init_gain)
            elif init_type == "xavier":
                init.xavier_normal_(m.weight.data, gain=init_gain)
            elif init_type == "kaiming":
                init.kaiming_normal_(m.weight.data, a=0, mode="fan_in")
            elif init_type == "orthogonal":
                init.orthogonal_(m.weight.data, gain=init_gain)
            if hasattr(m, "bias") and m.bias is not None:
                init.constant_(m.bias.data, 0.0)
        elif classname.find("BatchNorm2d") != -1:
            init.normal_(m.weight.data, 1.0, init_gain)
            init.constant_(m.bias.data, 0.0)
    net.apply(init_func)

def init_net(net, init_type="normal", init_gain=0.02):
    import os
    if torch.cuda.is_available():
        if "LOCAL_RANK" in os.environ:
            local_rank = int(os.environ["LOCAL_RANK"])
            net.to(local_rank)
        else:
            net.to(0)
    init_weights(net, init_type, init_gain=init_gain)
    return net


class UnetSkipConnectionBlock(nn.Module):
    def __init__(self, outer_nc, inner_nc, input_nc=None, submodule=None, outermost=False, innermost=False, norm_layer=nn.BatchNorm2d, use_dropout=False):
        super(UnetSkipConnectionBlock, self).__init__()
        self.outermost = outermost
        use_bias = (norm_layer.func == nn.InstanceNorm2d) if type(norm_layer) == functools.partial else (norm_layer == nn.InstanceNorm2d)
        if input_nc is None:
            input_nc = outer_nc
        downconv = nn.Conv2d(input_nc, inner_nc, kernel_size=4, stride=2, padding=1, bias=use_bias)
        downrelu = nn.LeakyReLU(0.2, True)
        downnorm = norm_layer(inner_nc)
        uprelu = nn.ReLU(True)
        upnorm = norm_layer(outer_nc)

        if outermost:
            upconv = nn.ConvTranspose2d(inner_nc * 2, outer_nc, kernel_size=4, stride=2, padding=1)
            down = [downconv]
            up = [uprelu, upconv, nn.Tanh()]
            model = down + [submodule] + up
        elif innermost:
            upconv = nn.ConvTranspose2d(inner_nc, outer_nc, kernel_size=4, stride=2, padding=1, bias=use_bias)
            down = [downrelu, downconv]
            up = [uprelu, upconv, upnorm]
            model = down + up
        else:
            upconv = nn.ConvTranspose2d(inner_nc * 2, outer_nc, kernel_size=4, stride=2, padding=1, bias=use_bias)
            down = [downrelu, downconv, downnorm]
            up = [uprelu, upconv, upnorm]
            if use_dropout:
                model = down + [submodule] + up + [nn.Dropout(0.5)]
            else:
                model = down + [submodule] + up

        self.model = nn.Sequential(*model)

    def forward(self, x):
        if self.outermost:
            return self.model(x)
        else:
            return torch.cat([x, self.model(x)], 1)

class UnetGenerator(nn.Module):
    def __init__(self, input_nc, output_nc, num_downs, ngf=64, norm_layer=nn.BatchNorm2d, use_dropout=False):
        super(UnetGenerator, self).__init__()
        unet_block = UnetSkipConnectionBlock(ngf * 8, ngf * 8, input_nc=None, submodule=None, norm_layer=norm_layer, innermost=True)
        for i in range(num_downs - 5):
            unet_block = UnetSkipConnectionBlock(ngf * 8, ngf * 8, input_nc=None, submodule=unet_block, norm_layer=norm_layer, use_dropout=use_dropout)
        unet_block = UnetSkipConnectionBlock(ngf * 4, ngf * 8, input_nc=None, submodule=unet_block, norm_layer=norm_layer)
        unet_block = UnetSkipConnectionBlock(ngf * 2, ngf * 4, input_nc=None, submodule=unet_block, norm_layer=norm_layer)
        unet_block = UnetSkipConnectionBlock(ngf, ngf * 2, input_nc=None, submodule=unet_block, norm_layer=norm_layer)
        self.model = UnetSkipConnectionBlock(output_nc, ngf, input_nc=input_nc, submodule=unet_block, outermost=True, norm_layer=norm_layer)

    def forward(self, input):
        return self.model(input)

###############################################################################
# 3. CLASS MODEL ĐIỀU KHIỂN (Logic Train)
###############################################################################
class UnetModel(BaseModel):
    @staticmethod
    def modify_commandline_options(parser, is_train=True):
        parser.set_defaults(dataset_mode='aligned', netG='unet_256')
        if is_train:
            parser.add_argument('--lambda_L1', type=float, default=100.0, help='weight for L1 loss')
        return parser

    def __init__(self, opt):
        BaseModel.__init__(self, opt)
        self.loss_names = ['G_L1']
        self.visual_names = ['real_A', 'fake_B', 'real_B']
        self.model_names = ['G']
        
        norm_layer = get_norm_layer(norm_type=opt.norm)
        num_downs = 8 if opt.netG == 'unet_256' else 7
        
        unet = UnetGenerator(
            opt.input_nc,
            opt.output_nc,
            num_downs,
            opt.ngf,
            norm_layer=norm_layer,
            use_dropout=not opt.no_dropout
        )
        
        self.netG = init_net(unet, opt.init_type, opt.init_gain)

        if self.isTrain:
            self.criterionL1 = nn.L1Loss()
            self.optimizer_G = torch.optim.Adam(
                self.netG.parameters(),
                lr=opt.lr,
                betas=(opt.beta1, 0.999)
            )
            self.optimizers.append(self.optimizer_G)

    def set_input(self, input):
        AtoB = self.opt.direction == 'AtoB'
        self.real_A = input['A' if AtoB else 'B'].to(self.device)
        self.real_B = input['B' if AtoB else 'A'].to(self.device)
        self.image_paths = input['A_paths' if AtoB else 'B_paths']

    def forward(self):
        self.fake_B = self.netG(self.real_A)

    def backward_G(self):
        self.loss_G_L1 = self.criterionL1(
            self.fake_B,
            self.real_B
        ) * self.opt.lambda_L1

        self.loss_G_L1.backward()

    def optimize_parameters(self):
        self.forward()
        self.optimizer_G.zero_grad()
        self.backward_G()
        self.optimizer_G.step()
'''

# Ghi file
with open(file_path, "w", encoding="utf-8") as f:
    f.write(content)

print(f"Đã tạo file: {file_path}")

Đã tạo file: models/unet_model.py


In [6]:
import os

# Đường dẫn file cấu hình bạn muốn tạo
file_path = 'configs/model1_unet.yaml'

# Tự động tạo thư mục 'configs' nếu nó chưa tồn tại
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Tiến hành ghi nội dung vào file yaml
with open(file_path, 'w', encoding='utf-8') as f:
    f.write("""dataroot: /kaggle/input/sketch2image-dataset/cufs
dataset_mode: aligned
direction: AtoB
load_size: 128
crop_size: 128
preprocess: resize_and_crop
no_flip: true
num_threads: 4

model: unet
netG: resnet_6blocks
netD: pixel
norm: none
no_dropout: true
init_type: normal
init_gain: 0.01
gan_mode: lsgan

lambda_L1: 60

batch_size: 16
n_epochs: 100
n_epochs_decay: 100
lr: 0.0001
lr_policy: linear
beta1: 0.7

seed: 2026

save_latest_freq: 5000
save_epoch_freq: 5
save_by_iter: false

display_freq: 400
print_freq: 100

num_test: 100

name: cufs_ablation_j_resnet6_pixel_lsgan_normnone
checkpoints_dir: ./checkpoints
""")

print(f"Đã tạo thư mục và ghi file thành công tại: {file_path}")

Đã tạo thư mục và ghi file thành công tại: configs/model1_unet.yaml


In [7]:
!pip install "numpy<2" --upgrade

In [8]:
import yaml

# Đọc file cấu hình yaml bạn vừa ghi ở trên
with open('configs/model1_unet.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

# Kiểm tra xem đã load đúng chưa
print("Batch size hiện tại:", cfg['batch_size'])

Batch size hiện tại: 16


In [9]:
with open('/kaggle/working/Sketch-to-Image-by-Pix2Pix/util/evaluator.py', 'r+') as f:
    content = f.read()
    f.seek(0, 0)
    # Chèn dòng import vào đầu file rồi ghi đè lại nội dung cũ
    f.write('from typing import Optional\n' + content)
print("Đã tự động sửa xong lỗi Optional!")

Đã tự động sửa xong lỗi Optional!


In [10]:
!python train.py \
  --dataroot {cfg['dataroot']} \
  --name {cfg['name']} \
  --model {cfg['model']} \
  --netG {cfg['netG']} \
  --dataset_mode {cfg['dataset_mode']} \
  --direction {cfg['direction']} \
  --gan_mode {cfg['gan_mode']} \
  --lambda_L1 {cfg['lambda_L1']} \
  --batch_size {cfg['batch_size']} \
  --lr {cfg['lr']} \
  --beta1 {cfg['beta1']} \
  --n_epochs {cfg['n_epochs']} \
  --n_epochs_decay {cfg['n_epochs_decay']} \
  --num_threads {cfg['num_threads']} \
  --preprocess {cfg['preprocess']} \
  --load_size {cfg['load_size']} \
  --crop_size {cfg['crop_size']} \
  --print_freq {cfg['print_freq']} \
  --display_freq {cfg['display_freq']} \
  --save_latest_freq {cfg['save_latest_freq']} \
  --save_epoch_freq {cfg['save_epoch_freq']} \
  --checkpoints_dir {cfg['checkpoints_dir']} 

----------------- Options ---------------
               batch_size: 16                            	[default: 1]
                    beta1: 0.7                           	[default: 0.5]
          checkpoints_dir: ./checkpoints                 
           continue_train: False                         
                crop_size: 128                           	[default: 256]
                 dataroot: /kaggle/input/sketch2image-dataset/cufs	[default: None]
             dataset_mode: aligned                       
                direction: AtoB                          
             display_freq: 400                           
          display_winsize: 256                           
                    epoch: latest                        
              epoch_count: 1                             
          eval_batch_size: 1                             
                eval_flip: False                         
     eval_kid_subset_size: 50                            
         eval_kid_su

# Testing

-   `python test.py --dataroot ./datasets/facades --direction BtoA --model pix2pix --name facades_unet256_lambda100_0610`

Change the `--dataroot`, `--name`, and `--direction` to be consistent with your trained model's configuration and how you want to transform images.

> Note that we specified --direction BtoA as Facades dataset's A to B direction is photos to labels.
> If you would like to apply a pre-trained model to a collection of input images (rather than image pairs), please use --model test option. See ./scripts/test_single.sh for how to apply a model to Facade label maps (stored in the directory facades/testB).

In [11]:
!python test.py \
  --dataroot {cfg['dataroot']} \
  --name {cfg['name']} \
  --model {cfg['model']} \
  --dataset_mode {cfg['dataset_mode']} \
  --direction {cfg['direction']} \
  --preprocess {cfg['preprocess']} \
  --load_size {int(cfg['load_size'])} \
  --crop_size {int(cfg['crop_size'])} \
  --num_test {int(cfg['num_test'])} \



----------------- Options ---------------
             aspect_ratio: 1.0                           
               batch_size: 1                             
          checkpoints_dir: ./checkpoints                 
                crop_size: 128                           	[default: 256]
                 dataroot: /kaggle/input/sketch2image-dataset/cufs	[default: None]
             dataset_mode: aligned                       
                direction: AtoB                          
          display_winsize: 256                           
                    epoch: latest                        
                     eval: False                         
                init_gain: 0.02                          
                init_type: normal                        
                 input_nc: 3                             
                  isTrain: False                         	[default: None]
                load_iter: 0                             	[default: 0]
                loa

# Visualize

In [12]:
import os
from pathlib import Path
import matplotlib.pyplot as plt

results_root = Path("./results") / cfg["name"]
if not results_root.exists():
    raise FileNotFoundError(f"Không thấy folder: {results_root}")

# Tìm mọi folder images trong run này
image_dirs = sorted(results_root.glob("**/images"))
if not image_dirs:
    raise FileNotFoundError(f"Không thấy thư mục images bên trong: {results_root}")

# Ưu tiên test_latest/images nếu có, không thì lấy folder images mới nhất
preferred = results_root / "test_latest" / "images"
base = preferred if preferred.exists() else max(image_dirs, key=lambda p: p.stat().st_mtime)

print("Using images dir:", base)

all_files = sorted([p.name for p in base.iterdir() if p.is_file()])
real_A_files = [f for f in all_files if f.endswith("_real_A.png")]

if not real_A_files:
    print("Không tìm thấy *_real_A.png trong", base)
else:
    samples = [f.replace("_real_A.png", "") for f in real_A_files[:3]]
    for sample in samples:
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        titles = ["Input (Sketch)", "Generated (Fake)", "Ground Truth (Real)"]
        files = [f"{sample}_real_A.png", f"{sample}_fake_B.png", f"{sample}_real_B.png"]
        for ax, title, fname in zip(axes, titles, files):
            img_path = base / fname
            if not img_path.exists():
                ax.set_title(f"Missing: {fname}")
                ax.axis("off")
                continue
            ax.imshow(plt.imread(img_path))
            ax.set_title(title)
            ax.axis("off")
        plt.suptitle(sample, fontsize=9, y=1.01)
        plt.tight_layout()
        plt.show()

FileNotFoundError: Không thấy folder: results/cufs_ablation_j_resnet6_pixel_lsgan_normnone